In [6]:
import pandas as pd
import re

In [37]:
df = pd.read_csv("./data/healthcare_dataset.csv")
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [52]:
import pandas as pd
from pymongo import MongoClient


# Connexion à MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client['healthcare_db']
collection = db['patients']

In [53]:
df = pd.read_csv("./data/healthcare_dataset.csv", on_bad_lines='skip')
df['Name'] = df['Name'].str.title()
df['Gender'] = df['Gender'].str.capitalize()
df['Blood Type'] = df['Blood Type'].str.upper()
df['Medical Condition'] = df['Medical Condition'].str.capitalize()
df['Doctor'] = df['Doctor'].str.title()
df['Hospital'] = df['Hospital'].str.replace('"', '').str.strip().str.title()
df['Insurance Provider'] = df['Insurance Provider'].str.title()
df['Admission Type'] = df['Admission Type'].str.capitalize()
df['Medication'] = df['Medication'].str.capitalize()
df['Test Results'] = df['Test Results'].str.capitalize()
df['Billing Amount'] = pd.to_numeric(df['Billing Amount'], errors='coerce')
df['Room Number'] = pd.to_numeric(df['Room Number'], errors='coerce')
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'], errors='coerce')
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'], errors='coerce')
df['Length of Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days
df = df.dropna(subset=['Name', 'Age', 'Gender', 'Medical Condition', 'Billing Amount'])
df = df.rename(columns=lambda c: re.sub(r"[ .-]+", "_", c.strip().lower()))
df.head()

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,length_of_stay
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,2
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,15
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal,30
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal,20


In [54]:
# Convertir en liste de dicts et insérer
records = df.to_dict('records')
collection.insert_many(records)
print(f"{len(records)} documents insérés avec succès.")

55500 documents insérés avec succès.


In [56]:
collection.find

In [57]:
import pandas as pd
import re
from typing import List, Tuple, Optional

def process_data(
    data_path: str,
    string_columns: Optional[List[str]] = None,
    numeric_columns: Optional[List[str]] = None,
    date_columns: Optional[List[str]] = None,
    duration_columns: Optional[Tuple[str, str, str]] = None,
    required_columns: Optional[List[str]] = None,
    string_transform: str = 'title',
    on_bad_lines: str = 'skip'
) -> pd.DataFrame:
    """Process a CSV DataFrame with minimal loops using vectorized operations.

    Args:
        data_path: Path to CSV file.
        string_columns: Columns to treat as strings.
        numeric_columns: Columns to convert to numeric.
        date_columns: Columns to convert to datetime.
        duration_columns: (start_date, end_date, new_column) for duration in days.
        required_columns: Columns requiring non-NaN values.
        string_transform: String transformation ('title', 'capitalize', 'upper').
        on_bad_lines: How to handle bad CSV lines ('skip', 'warn', 'error').

    Returns:
        Processed DataFrame with cleaned and normalized columns.
    """
    # Charger CSV et normaliser les noms de colonnes
    df = pd.read_csv(data_path, on_bad_lines=on_bad_lines)
    df.columns = [re.sub(r"[ .-]+", "_", c.strip().lower()) for c in df.columns]

    # Détection automatique des types si non spécifiés
    string_columns = string_columns or df.select_dtypes('object').columns.tolist()
    numeric_columns = numeric_columns or df.select_dtypes(['int64', 'float64']).columns.tolist()
    date_columns = date_columns or [col for col in df.columns if 'date' in col.lower()]

    # Transformations vectorisées pour les chaînes
    if string_columns:
        transform = getattr(str, string_transform, str.title)
        str_cols = [col for col in string_columns if col in df.columns]
        df[str_cols] = df[str_cols].apply(lambda s: s.str.replace('"', '', regex=False).str.strip().apply(lambda x: transform(x) if isinstance(x, str) else x))

    # Conversions vectorisées pour les numériques et dates
    if numeric_columns:
        num_cols = [col for col in numeric_columns if col in df.columns]
        df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    if date_columns:
        date_cols = [col for col in date_columns if col in df.columns]
        df[date_cols] = df[date_cols].apply(pd.to_datetime, errors='coerce')

    # Calcul de la durée (seule boucle explicite, conditionnelle)
    if duration_columns and duration_columns[0] in df.columns and duration_columns[1] in df.columns:
        df[duration_columns[2]] = (df[duration_columns[1]] - df[duration_columns[0]]).dt.days

    # Suppression des NaN dans les colonnes requises
    if required_columns:
        df = df.dropna(subset=[col for col in required_columns if col in df.columns])

    return df

In [59]:
l = processed_data = process_data("./data/healthcare_dataset.csv")
l

,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,Adrienne Bell,43,Female,Ab+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,Elizabeth Jackson,42,Female,O+,Asthma,2020-08-16,Joshua Jarvis,Jones-Thompson,Blue Cross,2650.714952,417,Elective,2020-09-15,Penicillin,Abnormal
55496,Kyle Perez,61,Female,Ab-,Obesity,2020-01-23,Taylor Sullivan,Tucker-Moyer,Cigna,31457.797307,316,Elective,2020-02-01,Aspirin,Normal
55497,Heather Wang,38,Female,B+,Hypertension,2020-07-13,Joe Jacobs Dvm,"And Mahoney Johnson Vasquez,",Unitedhealthcare,27620.764717,347,Urgent,2020-08-10,Ibuprofen,Abnormal
55498,Jennifer Jones,43,Male,O-,Arthritis,2019-05-25,Kimberly Curry,"Jackson Todd And Castro,",Medicare,32451.092358,321,Elective,2019-05-31,Ibuprofen,Abnormal
